In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split

### EDA

In [2]:
df = pd.read_csv("../../datasets/dataset/train.csv")

In [3]:
print("length of dataset: ", len(df))
print("Labels distribution: ", df['label'].value_counts(normalize=True) * 100)
print("Total missing values: ", df.isnull().sum().sum())

length of dataset:  40000
Labels distribution:  label
0    96.8
1     3.2
Name: proportion, dtype: float64
Total missing values:  153603


In [4]:
from pandas.core.arrays import categorical


dense_cols = [f"integer_feature_{i}" for i in range(1, 14)]
categorical_cols = [f"categorical_feature_{i}" for i in range(1, 27)]

df[dense_cols] = df[dense_cols].fillna(0)

for col in dense_cols:
    df[col] = np.log(1 + df[col] - df[col].min() + 1)

df[dense_cols] = MinMaxScaler().fit_transform(df[dense_cols])

df[categorical_cols] = df[categorical_cols].fillna('UNK')
label_encoders_dict = {}
sparse_cardinalities = []

for col in categorical_cols:
    l_e = LabelEncoder()
    df[col] = l_e.fit_transform(df[col].astype(str))
    label_encoders_dict[col] = l_e  # <
    sparse_cardinalities.append(df[col].nunique())

    

In [5]:
class customDataset(Dataset):
    def __init__(self, data):
        self.dense = torch.tensor(data[dense_cols].values, dtype=torch.float32)
        self.categorical = torch.tensor(data[categorical_cols].values, dtype=torch.long)
        self.labels = torch.tensor(data['label'].values, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return self.dense[index], self.categorical[index], self.labels[index]

In [15]:

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

train_loader = DataLoader(customDataset(train_df), batch_size=1024, shuffle=True)
val_loader = DataLoader(customDataset(val_df), batch_size=1024, shuffle=False)


In [16]:
class NeuralCTR(nn.Module):
    def __init__(self, sparse_cardinalities, num_dense_features, embed_dim=16, hidden_dims=[256, 128, 64], dropout=0.2):
        super().__init__()

        self.embeddings = nn.ModuleList([
            nn.Embedding(num_embeddings=int(vocab_size), embedding_dim=embed_dim) for vocab_size in sparse_cardinalities
        ])

        total_input_dim = (len(sparse_cardinalities) * embed_dim) + num_dense_features

        layers = []
        input_dim = total_input_dim

        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(input_dim, hidden_dim))
            layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            input_dim = hidden_dim

        layers.append(nn.Linear(input_dim, 1))
        self.mlp = nn.Sequential(*layers)

    def forward(self, dense_x, categ_x):
        embeds = [
            self.embeddings[i](categ_x[:, i]) for i in range(categ_x.shape[1])   #*
        ]

        embeds_concat = torch.cat(embeds, dim=1)

        x = torch.cat([dense_x,  embeds_concat], dim=1)

        out = self.mlp(x)

        return out.squeeze()




    



In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = NeuralCTR(
    sparse_cardinalities=sparse_cardinalities, 
    num_dense_features=len(dense_cols)
).to(device)

print(model)

NeuralCTR(
  (embeddings): ModuleList(
    (0): Embedding(13191, 16)
    (1): Embedding(3757, 16)
    (2): Embedding(4968, 16)
    (3): Embedding(1851, 16)
    (4): Embedding(4253, 16)
    (5): Embedding(3, 16)
    (6): Embedding(3111, 16)
    (7): Embedding(741, 16)
    (8): Embedding(22, 16)
    (9): Embedding(11585, 16)
    (10): Embedding(5569, 16)
    (11): Embedding(7723, 16)
    (12): Embedding(9, 16)
    (13): Embedding(904, 16)
    (14): Embedding(2025, 16)
    (15): Embedding(43, 16)
    (16): Embedding(4, 16)
    (17): Embedding(341, 16)
    (18): Embedding(14, 16)
    (19): Embedding(13509, 16)
    (20): Embedding(9003, 16)
    (21): Embedding(12629, 16)
    (22): Embedding(5095, 16)
    (23): Embedding(3504, 16)
    (24): Embedding(36, 16)
    (25): Embedding(24, 16)
  )
  (mlp): Sequential(
    (0): Linear(in_features=429, out_features=256, bias=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout

In [ ]:
import torch.optim as optim
from sklearn.metrics import roc_auc_score, accuracy_score, average_precision_score, log_loss, f1_score

pos_weight = torch.tensor([24.0]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = optim.Adam(model.parameters(), lr = 0.001, weight_decay=1e-5)
epochs= 5

training_losses = []
val_losses = []

# begin training

for epoch in range(epochs):
    model.train()

    running_loss = 0.0

    for dense_x, categ_x, labels in train_loader:
        dense_x, categ_x, labels = dense_x.to(device), categ_x.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(dense_x, categ_x)
        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item() * labels.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)
    training_losses.append(epoch_loss)

    model.eval()
    running_loss = 0.

    all_preds = []

    all_labels = []

    with torch.no_grad():
        for dense_x, sparse_x, labels in val_loader:
            dense_x, sparse_x, labels = dense_x.to(device), sparse_x.to(device), labels.to(device)
            out = model(dense_x, sparse_x)

            loss = criterion(out, labels)

            running_loss += loss.item() * labels.size(0)

            probs = torch.sigmoid(out)
            all_preds.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
        
    epoch_valid_loss = running_loss / len(val_loader.dataset)
    val_losses.append(epoch_valid_loss)

    print(f"Epoch {epoch+1:02d}/{epochs} | Train Loss: {epoch_loss:.4f} | Val Loss: {epoch_valid_loss:.4f}")






    


Epoch 01/5 | Train Loss: 0.0776 | Val Loss: 0.1779
Epoch 02/5 | Train Loss: 0.0677 | Val Loss: 0.1933
Epoch 03/5 | Train Loss: 0.0580 | Val Loss: 0.2061
Epoch 04/5 | Train Loss: 0.0548 | Val Loss: 0.2212
Epoch 05/5 | Train Loss: 0.0485 | Val Loss: 0.2205


## Evaluations

In [23]:
all_labels = np.array(all_labels).astype(int)
binary_predxns = [1 if p >= 0.5 else 0 for p in all_preds]

roc_auc = roc_auc_score(all_labels, all_preds)
pr_auc = average_precision_score(all_labels, all_preds)
accuracy = accuracy_score(all_labels, binary_predxns)
logloss = log_loss(all_labels, all_preds)
f1 = f1_score(all_labels, binary_predxns)

print("ROC-AUC: ", roc_auc)
print("Precision Recall: ", pr_auc)
print("Accuracy: ", accuracy)
print("logloss: ", logloss)
print("f1 Scoe: ", f1)

ROC-AUC:  0.5956002227530992
Precision Recall:  0.0536325381497194
Accuracy:  0.963875
logloss:  0.22052151164635547
f1 Scoe:  0.0462046204620462
